# 📘 Session 5: Retrieval-Augmented Generation (RAG) & Document Intelligence

**Core Concepts Covered:**
- **Document Loaders:** Comparing deep document parsing with **Docling** (OCR, layout & table detection) vs fast markdown extraction with **PyMuPDF4LLM**.
- **Text Splitting & Chunking:** Preserving semantic structure and context overlap using `RecursiveCharacterTextSplitter`.
- **Vector Embeddings:** Generating dense semantic vector representations using Google Gemini (`gemini-embedding-2`).
- **Vector Database (Qdrant):** Creating managed cloud collections, upserting chunk payloads, and performing similarity searches with cosine distance scoring.
- **RAG Tool & Agent Integration:** Packaging semantic search into a LangChain `@tool` decorated with LangSmith `@traceable`, and grounding an autonomous agent to answer questions from documents.


## 1. Environment Setup & Configuration
Load API keys and LangSmith tracing configuration from `.env`.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv(override=True)

print("LangSmith Tracing:", os.getenv("LANGSMITH_TRACING"))
print("LangSmith Project:", os.getenv("LANGSMITH_PROJECT"))


## 2. Advanced Document Loading
### A. Docling: Deep Layout & Table Parser
Docling utilizes specialized AI models for layout detection, reading order, and table structure reconstruction.

In [ ]:
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_docling.loader import DoclingLoader

FILE_PATH = "paper.pdf"

# Configure pipeline: disable OCR for digital PDFs to accelerate loading
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

docling_loader = DoclingLoader(file_path=FILE_PATH, converter=converter)
docling_docs = docling_loader.load()

print(f"Docling parsed {len(docling_docs)} document element(s).")
print("Sample content preview:")
print(docling_docs[0].page_content[:300])


### B. PyMuPDF4LLM: High-Speed Markdown Loader
Fast markdown extraction retaining page boundaries, tables, and document metadata.

In [ ]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

loader = PyMuPDF4LLMLoader(
    file_path="paper.pdf",
    mode="single",
    pages_delimiter="\n\f"
)

docs = loader.load()
print(f"PyMuPDF4LLM loaded {len(docs)} document(s).")
print("Metadata:", docs[0].metadata)
print("\nFirst 200 characters of extracted Markdown:")
print(docs[0].page_content[:200])


## 3. Text Chunking Strategies
Chunking breaks lengthy documents into manageable passages. `RecursiveCharacterTextSplitter` respects natural text boundaries (paragraphs, sentences) while maintaining a sliding overlap.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

sample_text = """Text is naturally organized into hierarchical units such as paragraphs, sentences, and words. We can leverage this inherent structure to inform our splitting strategy, creating splits that maintain natural language flow, maintain semantic coherence within split, and adapt to varying levels of text granularity. LangChain’s RecursiveCharacterTextSplitter implements this concept:
The RecursiveCharacterTextSplitter attempts to keep larger units (e.g., paragraphs) intact.
If a unit exceeds the chunk size, it moves to the next level (e.g., sentences).
This process continues down to the word level if necessary."""

demo_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=15)
sample_chunks = demo_splitter.split_text(sample_text)

print(f"Split sample text into {len(sample_chunks)} chunks.")
for i, chunk in enumerate(sample_chunks):
    print(f"Chunk {i+1}: {chunk}")


## 4. Vector Embeddings (Google Gemini)
Embeddings map text into a continuous high-dimensional vector space where semantically related concepts reside close together.

In [ ]:
from langchain_core.documents import Document
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Initialize Google Gemini embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-2")

test_docs = [
    Document(page_content="Dogs are great companions, known for their loyalty and friendliness.", metadata={"source": "pets"}),
    Document(page_content="Cats are independent pets that often enjoy their own space.", metadata={"source": "pets"}),
]

vector_1 = embeddings.embed_query(test_docs[0].page_content)
vector_2 = embeddings.embed_query(test_docs[1].page_content)

print(f"Embedding dimension: {len(vector_1)}")
print(f"First 5 components of vector 1: {vector_1[:5]}")


## 5. Qdrant Cloud Vector Database Setup
Connect to Qdrant Cloud and ensure the target collection exists with matching vector dimensions and Cosine distance.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

qdrant_url = os.getenv("QDRANT_URL", "https://9733c34f-1938-43d1-9633-4d9859902cb2.eu-west-2-0.aws.cloud.qdrant.io")
qdrant_api_key = os.getenv("QDRANT_API_KEY")

# Initialize client with safe 60s timeout to avoid WriteTimeout during cloud upsert
client = QdrantClient(url=qdrant_url, api_key=qdrant_api_key, timeout=60)

COLLECTION_NAME = "research_papers"
vector_size = len(embeddings.embed_query("test dimension"))

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE)
    )
    print(f"Collection '{COLLECTION_NAME}' created.")
else:
    print(f"Collection '{COLLECTION_NAME}' already exists.")

vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings,
)


## 6. Document Chunking & Indexing into Qdrant
Split the research paper into 1000-character chunks with 200-character overlap and index them into Qdrant.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True
)

paper_splits = text_splitter.split_documents(docs)
print(f"Generated {len(paper_splits)} document chunks from paper.pdf.")

# Upsert chunks to Qdrant
doc_ids = vector_store.add_documents(documents=paper_splits)
print(f"Successfully indexed {len(doc_ids)} chunks into Qdrant collection '{COLLECTION_NAME}'.")


## 7. Semantic Similarity Search
Query the collection using cosine similarity to retrieve the most relevant passages alongside their similarity scores.

In [ ]:
query = "What does the paper abstract state regarding the proposed fusion framework?"
search_results = vector_store.similarity_search_with_score(query, k=2)

for rank, (result_doc, score) in enumerate(search_results, start=1):
    print(f"--- Match #{rank} (Cosine Similarity Score: {score:.4f}) ---")
    print(result_doc.page_content[:400])
    print("Metadata:", result_doc.metadata)
    print()


## 8. Building a Traceable RAG Tool
Wrap the similarity search into a LangChain `@tool` and decorate it with `@traceable` for automatic observability in LangSmith.

In [ ]:
from langchain_core.tools import tool
from langsmith import traceable

@tool
@traceable(name="query_paper_knowledge")
def query_paper_knowledge(query: str) -> str:
    """
    Search and retrieve factual excerpts from the research paper on ocular disease
    recognition, vessel segmentation, and multi-backbone feature fusion.
    """
    results = vector_store.similarity_search_with_score(query, k=3)
    formatted_chunks = []
    for doc, score in results:
        formatted_chunks.append(
            f"[Score: {score:.4f}]\n{doc.page_content}\nSource: {doc.metadata.get('source', 'paper.pdf')}"
        )
    return "\n\n---\n\n".join(formatted_chunks)

# Test the tool invocation
sample_output = query_paper_knowledge.invoke({"query": "what is the abstract"})
print(sample_output[:350])


## 9. Autonomous RAG Agent Execution
Assemble an autonomous agent equipped with our `query_paper_knowledge` tool. The agent automatically decides when to consult the vector database to answer user inquiries.

In [ ]:
from langchain.agents import create_agent

# Initialize agent with Gemini (or swap with ollama:llama3.2:3b if local server is active)
agent = create_agent(
    model="google_genai:gemini-2.5-flash",
    tools=[query_paper_knowledge]
)

user_question = "What accuracy did the proposed fusion model achieve on the hospital test set?"

response = agent.invoke({
    "messages": [{"role": "user", "content": user_question}]
})

print("User Question:", user_question)
print("\nAgent Answer:")
print(response["messages"][-1].content)
